In [0]:

%sql
CREATE SCHEMA IF NOT EXISTS medical_pipeline.silver;

In [0]:


from pyspark.sql.functions import col
import re
from pyspark.sql.functions import coalesce
from pyspark.sql.functions import try_to_date
from pyspark.sql.functions import lower, trim
from pyspark.sql.functions import regexp_replace

df_silver_patients = spark.table("medical_pipeline.bronze.patients")

df_silver_patients =df_silver_patients.select("id", "prefix", "first", "last", "suffix", "maiden", "birthdate_" , "death_date", "gender", "race", "ethnicity" , "state", "zip_code", "lat", "lon", "county", "marital_status")

df_silver_patients = df_silver_patients.withColumn("birthdate_", coalesce(try_to_date("birthdate_", "yyyy-MM-dd"), try_to_date("birthdate_", "dd-MM-yyyy"))) \
                         .withColumn("death_date",coalesce(try_to_date("death_date", "yyyy-MM-dd"), try_to_date("death_date", "dd-MM-yyyy"))) \
                         .withColumn("zip_code", col("zip_code").cast("string")) \
                         .withColumn("lat", col("lat").cast("double")) \
                         .withColumn("lon", col("lon").cast("double"))


df_silver_patients = df_silver_patients.fillna({ "maiden": "NA", "suffix": "NA", "prefix": "NA" })

df_silver_patients = df_silver_patients.dropDuplicates(["id"])


df_silver_patients = df_silver_patients.withColumn("gender", lower(trim(col("gender")))) \
                         .withColumn("race", lower(trim(col("race")))) \
                         .withColumn("ethnicity", lower(trim(col("ethnicity")))) \
                         .withColumn("marital_status", lower(trim(col("marital_status"))))

df_silver_patients = df_silver_patients.filter(col("id").isNotNull())


df_silver_patients = df_silver_patients.withColumn("first", regexp_replace("first", r"\d+", "")) \
.withColumn("last", regexp_replace("last", r"\d+", "")).withColumn("maiden", regexp_replace("maiden", r"\d+", ""))

df_silver_patients = df_silver_patients.withColumn("county",regexp_replace("county", r"(?i)\s*county\s*$", ""))


In [0]:
 df_silver_patients.write.mode("overwrite").saveAsTable("medical_pipeline.silver.patients")